# 01 — Scrape MakerWorld

| | |
|---|---|
| **Pipeline stages** | `validate` → `scrape` |
| **Service module** | `backend.services.pipeline.scrape_makerworld` |
| **Website equivalent** | `POST /api/import/stream` (first two progress events) |
| **Next notebook** | `02_extract_bom.ipynb` (`extract_bom` stage) |

Uses the **same code** as the web app. Run cell 1 first (`prepare_crawl_env`).

- Single crawl via `safe_scrape()` (90s timeout, no hang)
- No second `fetch_page` — inspect `ScrapeResult` only
- Full regression loop is **off** unless you set `RUN_REGRESSION_ALL = True`

In [ ]:
from backend.notebook_utils import (
    load_regression_catalog,
    notebook_progress,
    pick_sample_url,
    prepare_crawl_env,
    print_pipeline_map,
    safe_scrape,
)

prepare_crawl_env(scraper="auto")  # scraper="playwright" if 403
print_pipeline_map()

# Set True to scrape every URL in data/regression_urls.json (slow)
RUN_REGRESSION_ALL = False

progress = notebook_progress("01")
SAMPLE = pick_sample_url(index=1, require_bom=True)
SAMPLE_URL = SAMPLE["url"]
print("\nSample:", SAMPLE["label"])
print(SAMPLE_URL)

In [ ]:
result = await safe_scrape(SAMPLE_URL, on_progress=progress, timeout_s=90)

if result is None:
    print("No ScrapeResult — fix network/scraper or try playwright, then re-run.")
else:
    print("\n--- ScrapeResult (same object the API uses) ---")
    print("title:", result.title)
    print("bom_source:", result.bom_source)
    print("bom_filename:", result.bom_filename)
    print("embedded_parts:", len(result.embedded_parts))
    print("description_parts:", len(result.description_parts))
    if result.bom_bytes:
        print("bom_bytes:", len(result.bom_bytes), "bytes")
    if result.warnings:
        for w in result.warnings:
            print("warning:", w)
    if result.embedded_parts:
        print("\nFirst embedded part:", result.embedded_parts[0].original_name)
    if result.description_parts:
        print("First description part:", result.description_parts[0].original_name)

In [ ]:
if not RUN_REGRESSION_ALL:
    print("Skipping regression loop (set RUN_REGRESSION_ALL = True in cell 1 to enable).")
else:
    catalog = load_regression_catalog()
    for entry in catalog.get("urls", []):
        label = entry.get("label", entry["url"])[:40]
        p = notebook_progress(label)
        r = await safe_scrape(entry["url"], on_progress=p, timeout_s=90)
        if r is None:
            print("  → FAILED\n")
        else:
            print(
                f"  → bom_source={r.bom_source} "
                f"embedded={len(r.embedded_parts)} "
                f"desc={len(r.description_parts)} "
                f"file={r.bom_filename}\n"
            )